In [ ]:
#Optical Flow and Motion Estimation

import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from google.colab import files

# ==== Upload Video ====
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# === Open video capture ===
cap = cv2.VideoCapture(video_path)

# Use webcam (if running locally, not Colab)
# cap = cv2.VideoCapture(0)

# === Parameters for Lucas-Kanade Optical Flow ===
lk_params = dict(winSize  = (15, 15),
                 maxLevel = 2,
                 criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# === Parameters for Shi-Tomasi corner detection ===
feature_params = dict(maxCorners = 100,
                      qualityLevel = 0.3,
                      minDistance = 7,
                      blockSize = 7)

# === Take first frame and find corners in it ===
ret, old_frame = cap.read()
if not ret:
    print("Failed to read video")
    cap.release()
    exit()

old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

# === Create a mask image for drawing purposes ===
mask = np.zeros_like(old_frame)

frame_count = 0
max_frames = 100  # Limit for demo

while cap.isOpened() and frame_count < max_frames:
    ret, frame = cap.read()
    if not ret:
        break

    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # === Calculate Optical Flow ===
    p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

    # === Select good points ===
    if p1 is not None:
        good_new = p1[st == 1]
        good_old = p0[st == 1]

        # === Draw motion vectors ===
        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new.ravel()
            c, d = old.ravel()
            mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), (0, 255, 0), 2)
            frame = cv2.circle(frame, (int(a), int(b)), 5, (0, 0, 255), -1)

        img = cv2.add(frame, mask)
    else:
        img = frame

    # === Show the frame in Colab ===
    cv2_imshow(img)
    key = cv2.waitKey(50) & 0xFF
    if key == 27:  # ESC to quit
        break

    # === Update previous frame and points ===
    old_gray = frame_gray.copy()
    p0 = good_new.reshape(-1, 1, 2)

    frame_count += 1

cap.release()
cv2.destroyAllWindows()
print("Done.")

In [ ]:
import cv2
import numpy as np
from google.colab import files

# ==== Upload Video ====
#print("Upload your video file (e.g., road_traffic.mp4)...")
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# === Open video capture ===
cap = cv2.VideoCapture(video_path)

# === Parameters for Lucas-Kanade Optical Flow ===
lk_params = dict(winSize  = (15, 15),
                 maxLevel = 2,
                 criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# === Parameters for Shi-Tomasi corner detection ===
feature_params = dict(maxCorners = 100,
                      qualityLevel = 0.3,
                      minDistance = 7,
                      blockSize = 7)

# === Take first frame and find corners in it ===
ret, old_frame = cap.read()
if not ret:
    print("Failed to read video.")
    cap.release()
    exit()

old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

# === Create a mask image for drawing motion vectors ===
mask = np.zeros_like(old_frame)

# === Prepare VideoWriter to save output video ===
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    fps = 20  # fallback frame rate if not detected

fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for mp4
out = cv2.VideoWriter('output.mp4', fourcc, fps, (frame_width, frame_height))

# === Process all frames ===
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # === Calculate Optical Flow ===
    p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

    # === Select good points ===
    if p1 is not None and st is not None:
        good_new = p1[st == 1]
        good_old = p0[st == 1]

        # === Draw motion vectors ===
        for new, old in zip(good_new, good_old):
            a, b = new.ravel()
            c, d = old.ravel()
            mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), (0, 255, 0), 2)
            frame = cv2.circle(frame, (int(a), int(b)), 5, (0, 0, 255), -1)

        img = cv2.add(frame, mask)

        # === Update for next iteration ===
        old_gray = frame_gray.copy()
        p0 = good_new.reshape(-1, 1, 2)
    else:
        img = frame

    # === Write processed frame to output video ===
    out.write(img)

# === Release resources ===
cap.release()
out.release()
cv2.destroyAllWindows()

print("✅ Processing complete. Output video saved as 'output.mp4'")

# === Offer the output video for download ===
files.download('output.mp4')

Saving road_trafifc.mp4 to road_trafifc (1).mp4
✅ Processing complete. Output video saved as 'output.mp4'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>